<a href="https://colab.research.google.com/github/arshdeepbangar/AAI2025/blob/main/Customer_churn_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Data source:
# Telecom Customer Churn Dataset
# https://github.com/bensadeghi/pyspark-churn-prediction

# Load the customer churn dataset
url = "https://raw.githubusercontent.com/bensadeghi/pyspark-churn-prediction/master/data/churn-bigml-80.csv"

df = pd.read_csv(url)

# Calculate total customer usage
df['total_usage_minutes'] = (
    df['Total day minutes']
    + df['Total eve minutes']
    + df['Total night minutes']
    + df['Total intl minutes']
)

# Calculate total customer charges
df['total_charge'] = (
    df['Total day charge']
    + df['Total eve charge']
    + df['Total night charge']
    + df['Total intl charge']
)

# Keep the columns needed for the model
df = df[
    [
        'Account length',
        'total_usage_minutes',
        'total_charge',
        'Customer service calls',
        'Area code',
        'Churn'
    ]
].copy()

# Rename columns
df.rename(columns={
    'Account length': 'account_length',
    'Customer service calls': 'customer_service_calls',
    'Area code': 'area_code',
    'Churn': 'churn'
}, inplace=True)

# Convert area code to categorical data
df['area_code'] = df['area_code'].astype(str)

# Convert churn into 1 and 0
df['churn'] = df['churn'].map({
    True: 1,
    False: 0,
    'True': 1,
    'False': 0
})

# Remove missing values
df.dropna(inplace=True)

# Convert churn to integer
df['churn'] = df['churn'].astype(int)

# Save cleaned dataset
df.to_csv('telecom_customer_churn.csv', index=False)

# Display dataset information
print("Dataset size:", df.shape)
print(df.head())

print("\nChurn counts:")
print(df['churn'].value_counts())

Dataset size: (2666, 6)
   account_length  total_usage_minutes  total_charge  customer_service_calls  \
0             128                717.2         75.56                       1   
1             107                625.2         59.24                       1   
2             137                539.4         62.29                       0   
3              84                564.8         66.80                       2   
4              75                512.0         52.09                       3   

  area_code  churn  
0       415      0  
1       415      0  
2       415      0  
3       408      0  
4       415      0  

Churn counts:
churn
0    2278
1     388
Name: count, dtype: int64


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


# Load the customer churn dataset
df = pd.read_csv('telecom_customer_churn.csv')


# Select numerical features
numerical_features = [
    'account_length',
    'total_usage_minutes',
    'total_charge',
    'customer_service_calls'
]

# Select categorical features
categorical_features = ['area_code']


# Select features and target
X = df[
    [
        'account_length',
        'total_usage_minutes',
        'total_charge',
        'customer_service_calls',
        'area_code'
    ]
]

y = df['churn']


# Scale numerical features and encode categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num',
         StandardScaler(),
         numerical_features),

        ('cat',
         OneHotEncoder(
             sparse_output=False,
             handle_unknown='ignore'
         ),
         categorical_features)
    ]
)


# Create the logistic regression model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        random_state=42,
        max_iter=1000
    ))
])


# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# Train the model
model.fit(X_train, y_train)


# Create a new customer
new_customer = pd.DataFrame({
    'account_length': [35],
    'total_usage_minutes': [600],
    'total_charge': [60],
    'customer_service_calls': [5],
    'area_code': ['415']
})


# Predict the probability that the customer will churn
churn_probability = model.predict_proba(new_customer)[0][1]


# Use a 0.5 threshold to classify churn
threshold = 0.5

churn_prediction = (
    1 if churn_probability >= threshold else 0
)


# Display prediction
print(
    f"Churn Probability for new customer: "
    f"{churn_probability:.2%}"
)

print(
    f"Churn Prediction "
    f"(1 = churn, 0 = no churn): "
    f"{churn_prediction}"
)


# Get categorical feature names
encoded_features = (
    model.named_steps['preprocessor']
    .named_transformers_['cat']
    .get_feature_names_out(categorical_features)
    .tolist()
)


# Combine all feature names
feature_names = (
    numerical_features
    + encoded_features
)


# Get model coefficients
coefficients = (
    model.named_steps['classifier']
    .coef_[0]
)


# Display model coefficients
print("\nModel Coefficients:")

for feature, coef in zip(
    feature_names,
    coefficients
):
    print(f"{feature}: {coef:.2f}")


# Calculate model accuracy
accuracy = model.score(
    X_test,
    y_test
)

print(
    f"\nModel Accuracy: "
    f"{accuracy:.2%}"
)

Churn Probability for new customer: 36.67%
Churn Prediction (1 = churn, 0 = no churn): 0

Model Coefficients:
account_length: 0.06
total_usage_minutes: -0.10
total_charge: 0.84
customer_service_calls: 0.61
area_code_408: 0.01
area_code_415: 0.03
area_code_510: -0.05

Model Accuracy: 84.83%


I used a Telecom Customer Churn dataset containing over 2,600 customer records. The purpose of this model was to predict whether a customer is likely to stop using the company's service, which is known as customer churn.

The features I used were account length, total usage minutes, total charges, customer service calls, and area code. The numerical features were scaled using StandardScaler so that features with larger numbers would not have more influence just because of their size. The area code was categorical data, so I used OneHotEncoder to convert it into numerical values that the model could understand.

I used logistic regression because the target has two possible outcomes: the customer either churns or does not churn. The model calculates a probability showing how likely a customer is to churn.

For the new customer, the model predicted a 36.67% probability of churn. I used a 0.5, or 50%, threshold to classify the customer. Since 36.67% is below 50%, the customer's prediction was 0, meaning the model does not currently classify this customer as likely to churn. If the probability had been 50% or higher, the customer would have been classified as 1, meaning they were considered at risk of churning.

The model coefficients help show which features are connected to a higher or lower chance of churn. A positive coefficient means the feature is associated with a higher chance of churn, while a negative coefficient means it is associated with a lower chance of churn.

In my model, total charge had a coefficient of 0.84, which was the strongest positive coefficient. Customer service calls had a coefficient of 0.61, showing that customers who require more customer service may also have a higher chance of leaving. Account length had a small positive coefficient of 0.06.

Total usage minutes had a coefficient of -0.10, meaning higher usage was associated with a slightly lower chance of churn in this model. The area code coefficients were very small: 408 was 0.01, 415 was 0.03, and 510 was -0.05, meaning area code had only a small effect on the model's churn prediction.

Because the numerical features were standardized before training, their coefficient sizes represent changes in the scaled features, rather than a direct change for one minute, one dollar, or one service call.
The model had an accuracy of 84.83% on the testing data. This means the model correctly classified about 85 out of every 100 customers in the test set.